# Notebook 01 — Mod30 Residue Manifold

**Bulletproof Colab/local version.**

This notebook builds a finite arithmetic analogue for the manifold-tiling behavior discussed in arXiv:2604.28119v1.

```text
integers → mod30 residue manifold
constraint gate → 8 persisting lanes
observable signal → local residue tiles
```


## 0. Bulletproof setup

Run this cell first. It supports:

- local repo execution
- `notebooks/` execution
- Colab after cloning/opening from GitHub
- loose notebook execution by creating a minimal `src/` fallback


In [ ]:

from pathlib import Path
import sys, os, textwrap

def find_repo_root(start=None, marker="src"):
    """
    Find a repo root by walking upward until a marker folder exists.
    Works when launched from:
    - repo root
    - notebooks/
    - Colab after cloning repo
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

# Colab/GitHub fallback:
# If opened as a loose notebook without repo files, create a minimal local repo layout
# so `from src.mod30 import ...` still works.
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""# src/mod30.py
from math import gcd

MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]

# finite constraint gate: residue filtering ≠ collapse to zero
# MOD210 = 2 * 3 * 5 * 7 = 210; phi(210) = 48 lanes
# MOD2310 = 2 * 3 * 5 * 7 * 11 = 2310; phi(2310) = 480 lanes
# MOD210_RESIDUES = [r for r in range(1, 210) if gcd(r, 210) == 1]
# MOD2310_RESIDUES = [r for r in range(1, 2310) if gcd(r, 2310) == 1]

def mod_index(n, mod):
    return n % mod

def mod_mask(n, residues, mod):
    return mod_index(n, mod) in residues

def generate_coprime_residues(mod):
    return [r for r in range(1, mod) if gcd(r, mod) == 1]

def mod30_index(n):
    return mod_index(n, MOD30)

def mod30_mask(n):
    return mod_mask(n, MOD30_RESIDUES, MOD30)

def mod30_residues(n_max):
    return [n for n in range(2, n_max) if mod30_mask(n)]
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""# src/tiling_metrics.py
from collections import Counter

def lane_counts(values, residue_fn):
    return dict(Counter(residue_fn(v) for v in values))

def lane_density(values, residue_fn):
    counts = lane_counts(values, residue_fn)
    total = sum(counts.values())
    if total == 0:
        return {}
    return {k: v / total for k, v in sorted(counts.items())}

def gate_summary(values, mask_fn):
    inside = [v for v in values if mask_fn(v)]
    outside = [v for v in values if not mask_fn(v)]
    total = len(values)
    return {
        "total": total,
        "inside": len(inside),
        "outside": len(outside),
        "inside_fraction": len(inside) / total if total else 0.0,
        "outside_fraction": len(outside) / total if total else 0.0,
    }
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""# src/plots.py
import matplotlib.pyplot as plt

def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"

for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())
print("sys.path[0]:", sys.path[0])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.mod30 import (
    MOD30,
    MOD30_RESIDUES,
    mod30_index,
    mod30_mask,
    mod30_residues,
    generate_coprime_residues,
)
from src.tiling_metrics import lane_density, gate_summary
from src.plots import save_current

print("MOD30:", MOD30)
print("MOD30_RESIDUES:", MOD30_RESIDUES)


## 1. Full mod30 residue manifold

Map integers to residue classes modulo 30.


In [ ]:
n_min = 1
n_max = 300
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))

df.head(12)


In [ ]:
plt.figure(figsize=(12, 4.5))
plt.scatter(df["n"], df["mod30_residue"], s=14)
plt.yticks(range(0, 30, 2))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Full mod30 residue manifold: all 30 residue states")
save_current(FIGURES_DIR / "01_full_mod30_residue_manifold.png")
plt.show()


## 2. Apply finite constraint gate

The mod30 gate keeps the classes coprime to 30:

```text
1, 7, 11, 13, 17, 19, 23, 29
```

This is a coarse wheel filter, not a primality test.


In [ ]:
summary = gate_summary(values.tolist(), mod30_mask)
summary


In [ ]:
inside = df[df["inside_mod30_gate"]]
outside = df[~df["inside_mod30_gate"]]

plt.figure(figsize=(12, 4.8))
plt.scatter(outside["n"], outside["mod30_residue"], s=10, alpha=0.35, label="outside gate")
plt.scatter(inside["n"], inside["mod30_residue"], s=22, label="inside mod30 gate")
plt.yticks(range(30))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Mod30 constraint gate: 8 local residue lanes persist")
plt.legend()
save_current(FIGURES_DIR / "02_mod30_gate_highlighted_lanes.png")
plt.show()


## 3. Lane density

The gate does not collapse the structure into one direction.  
It distributes candidate signal across 8 local lanes.


In [ ]:
density = lane_density(inside["n"].tolist(), mod30_index)
density_df = pd.DataFrame({
    "residue": list(density.keys()),
    "density_inside_gate": list(density.values()),
})
density_df


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(density_df["residue"], density_df["density_inside_gate"])
plt.xticks(MOD30_RESIDUES)
plt.xlabel("mod30 residue lane")
plt.ylabel("density among gated candidates")
plt.title("Eight mod30 residue classes act as local tiles whose union preserves the underlying structure.")
save_current(FIGURES_DIR / "03_mod30_lane_density.png")
plt.show()


## 4. Residue histogram: all residues vs gated lanes


In [ ]:
all_counts = df["mod30_residue"].value_counts().sort_index()
gate_counts = inside["mod30_residue"].value_counts().sort_index()

hist_df = pd.DataFrame({
    "residue": range(30),
    "all_count": [all_counts.get(r, 0) for r in range(30)],
    "gate_count": [gate_counts.get(r, 0) for r in range(30)],
    "inside_gate": [r in MOD30_RESIDUES for r in range(30)],
})

hist_df.to_csv(DATA_DIR / "01_mod30_residue_histogram.csv", index=False)
hist_df.head()


In [ ]:
plt.figure(figsize=(12, 4.8))
plt.bar(hist_df["residue"], hist_df["all_count"], alpha=0.35, label="all residues")
plt.bar(hist_df["residue"], hist_df["gate_count"], label="inside mod30 gate")
plt.xticks(range(30))
plt.xlabel("residue class mod 30")
plt.ylabel("count")
plt.title("Local residue tiles inside a finite mod30 manifold")
plt.legend()
save_current(FIGURES_DIR / "04_all_vs_gated_residue_histogram.png")
plt.show()


## 5. Bridge to arXiv:2604.28119v1

| Paper concept | Mod30 analogue |
|---|---|
| concept manifold | finite residue manifold |
| local tiling | persisting residue lanes |
| feature dilution | structure distributed over multiple lanes |
| global direction failure | no single residue lane captures full structure |
| interpretable local detectors | lane-wise residue filters |

Core phrase:

```text
A clean structure can be globally simple while locally tiled.
```


## 6. Foreshadowing: mod210 and mod2310

```text
mod30   = 2 × 3 × 5          → 8 lanes
mod210  = 2 × 3 × 5 × 7      → 48 lanes
mod2310 = 2 × 3 × 5 × 7 × 11 → 480 lanes
```


In [ ]:
mod30_lanes = generate_coprime_residues(30)
mod210_lanes = generate_coprime_residues(210)
mod2310_lanes = generate_coprime_residues(2310)

primorial_df = pd.DataFrame({
    "modulus": [30, 210, 2310],
    "lane_count": [len(mod30_lanes), len(mod210_lanes), len(mod2310_lanes)],
    "interpretation": [
        "coarse finite tiling",
        "finer primorial tiling",
        "next refined primorial tiling",
    ],
})
primorial_df


In [ ]:
primorial_df.to_csv(DATA_DIR / "01_primorial_lane_counts.csv", index=False)

plt.figure(figsize=(7, 4.5))
plt.plot(primorial_df["modulus"].astype(str), primorial_df["lane_count"], marker="o")
plt.xlabel("primorial modulus")
plt.ylabel("coprime residue lanes")
plt.title("Foreshadowing refinement: mod30 → mod210 → mod2310")
save_current(FIGURES_DIR / "05_primorial_lane_count_refinement.png")
plt.show()


## 7. Save compact summary


In [ ]:
summary_md = f"""# Notebook 01 Summary — Mod30 Residue Manifold

This notebook builds a finite arithmetic analogue for local manifold tiling.

## Core result

- Mod30 has 30 residue states.
- Excluding divisibility by 2, 3, and 5 leaves 8 residue lanes:
  `{MOD30_RESIDUES}`
- These lanes tile candidate structure locally rather than collapsing it into one global direction.

## arXiv:2604.28119v1 bridge

Sparse feature systems may tile concept manifolds locally.
Mod30 gives a transparent finite analogue:

- finite manifold: residues modulo 30
- local tiles: coprime residue lanes
- dilution: structure distributed across multiple lanes
"""

summary_path = OUTPUTS_DIR / "01_mod30_residue_manifold_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 8. Optional: zip-download pattern

Uncomment when running in Colab and you want one downloadable output bundle.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_01_mod30_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 9. Recommended Notebook 02

```text
02_local_tiling_vs_global_capture.ipynb
```

Goal:

- compare one global mod30 signal against 8 local lane detectors
- show why local tiled capture can preserve structure better than one global summary
- connect more explicitly to SAE feature fragmentation / dilution
```
